# Cancer Mortality Rate Prediction

A reproducible experimental comparison of linear regression, fully connected neural networks, ensemble trees, and feature-engineered CatBoost for predicting age-adjusted cancer mortality rates.

The notebook focuses on the modeling progression, feature engineering, validation strategy, and the generalization lessons revealed by the final public/private leaderboard results.

> The dataset files are intentionally not bundled with this repository. Place `train.csv`, `test.csv`, and `sample_submission.csv` in the repository's `data/` directory before running the notebook.


## 1. Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

RANDOM_STATE = 42


## 2. Load and inspect the data


In [ ]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Training shape:", train.shape)
print("Test shape:", test.shape)
print("\nMissing values:")
print(train.isna().sum()[train.isna().sum() > 0].sort_values(ascending=False))

train.head()


## 3. Controlled preprocessing

For the baseline and DNN comparison, `id`, `Geography`, and `binnedInc` are excluded. Numerical missing values are median-imputed using the training split only, and predictors are standardized using training-set statistics. The data are split 70/15/15 into training, validation, and local test sets.


In [ ]:
X = train.drop(columns=["TARGET_deathRate", "id", "Geography", "binnedInc"])
y = train["TARGET_deathRate"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train_imp)
X_val_scaled = x_scaler.transform(X_val_imp)
X_test_scaled = x_scaler.transform(X_test_imp)

print(X_train.shape, X_val.shape, X_test.shape)


## 4. Linear regression baseline


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

pred = linear_model.predict(X_test_scaled)
print("Test MSE:", mean_squared_error(y_test, pred))
print("Test R²:", r2_score(y_test, pred))


## 5. Deep neural networks

The target is standardized for DNN training to improve numerical stability with SGD. Predictions are inverse-transformed before evaluation on the original mortality-rate scale.


In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(RANDOM_STATE)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
y_val_scaled = y_scaler.transform(y_val.to_numpy().reshape(-1, 1)).ravel()

def build_dnn(hidden_units, dropout=0.0, loss="mse"):
    model = keras.Sequential([keras.layers.Input(shape=(X_train_scaled.shape[1],))])
    for units in hidden_units:
        model.add(keras.layers.Dense(units, activation="relu"))
        if dropout:
            model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(1))
    model.compile(optimizer="sgd", loss=loss)
    return model

def evaluate_dnn(hidden_units, dropout=0.0, loss="mse", epochs=100):
    model = build_dnn(hidden_units, dropout=dropout, loss=loss)
    history = model.fit(
        X_train_scaled, y_train_scaled,
        validation_data=(X_val_scaled, y_val_scaled),
        epochs=epochs, verbose=0
    )
    pred_scaled = model.predict(X_test_scaled, verbose=0)
    pred = y_scaler.inverse_transform(pred_scaled).ravel()
    return model, history, mean_squared_error(y_test, pred), r2_score(y_test, pred)

architectures = {
    "DNN-8": [8],
    "DNN-16-8": [16, 8],
    "DNN-16-8-4": [16, 8, 4],
    "DNN-30-16-8-4": [30, 16, 8, 4],
}

dnn_results = {}
histories = {}

for name, units in architectures.items():
    model, history, mse, r2 = evaluate_dnn(units)
    dnn_results[name] = {"MSE": mse, "R²": r2}
    histories[name] = history

drop_model, drop_history, drop_mse, drop_r2 = evaluate_dnn([16, 8, 4], dropout=0.2)
dnn_results["DNN-16-8-4 + Dropout"] = {"MSE": drop_mse, "R²": drop_r2}
histories["DNN-16-8-4 + Dropout"] = drop_history

pd.DataFrame(dnn_results).T


### Training curves


In [ ]:
plt.figure(figsize=(11, 6))
for name, history in histories.items():
    plt.plot(history.history["loss"], label=f"{name} train")
    plt.plot(history.history["val_loss"], linestyle="--", label=f"{name} validation")

plt.xlabel("Epoch")
plt.ylabel("MSE loss (standardized target)")
plt.title("DNN training and validation loss")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


### Alternative loss: MAE

MAE penalizes errors linearly, whereas MSE places greater weight on large errors. In the original experiment, the MSE-trained network performed slightly better on MSE/R² evaluation, which is consistent with its stronger emphasis on reducing larger residuals.


In [ ]:
mae_model, mae_history, mae_test_mse, mae_test_r2 = evaluate_dnn(
    [16, 8, 4], loss="mae"
)
print("MAE-trained DNN test MSE:", mae_test_mse)
print("MAE-trained DNN test R²:", mae_test_r2)


## 6. Tree-based baselines


In [ ]:
rf = RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_imp, y_train)
rf_pred = rf.predict(X_test_imp)

gb = GradientBoostingRegressor(
    n_estimators=400, learning_rate=0.1, max_depth=3, random_state=RANDOM_STATE
)
gb.fit(X_train_imp, y_train)
gb_pred = gb.predict(X_test_imp)

tree_results = pd.DataFrame({
    "Model": ["Random Forest", "Gradient Boosting"],
    "Test MSE": [
        mean_squared_error(y_test, rf_pred),
        mean_squared_error(y_test, gb_pred)
    ],
    "Test R²": [
        r2_score(y_test, rf_pred),
        r2_score(y_test, gb_pred)
    ]
})
tree_results


## 7. CatBoost with categorical information

For the extended experiment, `State` is extracted from `Geography`, while `binnedInc` is retained as a categorical feature. This is an expanded feature representation, so its performance should not be interpreted as an architecture-only comparison with the earlier baselines.


In [ ]:
from catboost import CatBoostRegressor

cat_df = train.copy()
cat_df["State"] = cat_df["Geography"].str.split(",").str[-1].str.strip()

X_cat = cat_df.drop(columns=["TARGET_deathRate", "id", "Geography"])
y_cat = cat_df["TARGET_deathRate"]
categorical_features = ["binnedInc", "State"]

for col in categorical_features:
    X_cat[col] = X_cat[col].fillna("Missing").astype(str)

X_train_cat, X_temp_cat, y_train_cat, y_temp_cat = train_test_split(
    X_cat, y_cat, test_size=0.30, random_state=RANDOM_STATE
)
X_val_cat, X_test_cat, y_val_cat, y_test_cat = train_test_split(
    X_temp_cat, y_temp_cat, test_size=0.50, random_state=RANDOM_STATE
)


## 8. Domain-informed feature engineering

`Deaths_per_100k` is a crude mortality-rate feature calculated only from predictors available in both train and test data:

`Deaths_per_100k = avgDeathsPerYear / popEst2015 × 100000`

It is related to, but not identical to, the age-adjusted target. In the original experiment, adding this single feature substantially improved held-out and cross-validation performance.


In [ ]:
def add_ratio_feature(X):
    X = X.copy()
    X["Deaths_per_100k"] = (
        X["avgDeathsPerYear"] / (X["popEst2015"] + 1e-8)
    ) * 100000
    return X

X_train_ratio = add_ratio_feature(X_train_cat)
X_val_ratio = add_ratio_feature(X_val_cat)
X_test_ratio = add_ratio_feature(X_test_cat)

ratio_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=0
)
ratio_model.fit(X_train_ratio, y_train_cat, cat_features=categorical_features)

ratio_pred = ratio_model.predict(X_test_ratio)
print("Local test MSE:", mean_squared_error(y_test_cat, ratio_pred))
print("Local test R²:", r2_score(y_test_cat, ratio_pred))


## 9. Five-fold cross-validation

Cross-validation is used to check whether the improvement from the engineered mortality-rate feature is consistent across folds rather than specific to a single split.


In [ ]:
cv_data = train.copy()
cv_data["State"] = cv_data["Geography"].str.split(",").str[-1].str.strip()

X_base = cv_data.drop(columns=["TARGET_deathRate", "id", "Geography"])
y_cv = cv_data["TARGET_deathRate"]

for col in categorical_features:
    X_base[col] = X_base[col].fillna("Missing").astype(str)

X_ratio_cv = add_ratio_feature(X_base)
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores = []
for tr_idx, va_idx in kf.split(X_ratio_cv):
    X_tr, X_va = X_ratio_cv.iloc[tr_idx], X_ratio_cv.iloc[va_idx]
    y_tr, y_va = y_cv.iloc[tr_idx], y_cv.iloc[va_idx]

    model = CatBoostRegressor(
        iterations=1000, learning_rate=0.05, depth=6,
        loss_function="RMSE", random_seed=RANDOM_STATE, verbose=0
    )
    model.fit(X_tr, y_tr, cat_features=categorical_features)
    scores.append(r2_score(y_va, model.predict(X_va)))

print("Fold R²:", np.round(scores, 4))
print("Mean R²:", np.mean(scores))
print("Std R²:", np.std(scores))


## 10. Train final model and create Kaggle submission

The final model is trained on all available labeled data with the selected feature representation. The original submission achieved **R² = 0.83616** on the Kaggle public leaderboard (30% of the hidden test data), but the final private leaderboard score on the remaining 70% was approximately **R² = 0.65**. Both scores are reported because the private result shows that the strong public score did not generalize to the larger hidden holdout.


In [ ]:
sample_submission = pd.read_csv("../data/sample_submission.csv")

train_final = train.copy()
test_final = test.copy()

for frame in (train_final, test_final):
    frame["State"] = frame["Geography"].str.split(",").str[-1].str.strip()
    frame["Deaths_per_100k"] = (
        frame["avgDeathsPerYear"] / (frame["popEst2015"] + 1e-8)
    ) * 100000

X_final = train_final.drop(columns=["TARGET_deathRate", "id", "Geography"])
y_final = train_final["TARGET_deathRate"]
X_kaggle = test_final.drop(columns=["id", "Geography"])

for col in categorical_features:
    X_final[col] = X_final[col].fillna("Missing").astype(str)
    X_kaggle[col] = X_kaggle[col].fillna("Missing").astype(str)

final_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=0
)
final_model.fit(X_final, y_final, cat_features=categorical_features)

submission = sample_submission.copy()
submission["TARGET_deathRate"] = final_model.predict(X_kaggle)
submission.to_csv("../submission.csv", index=False)

final_model.save_model("../models/best_catboost_deaths_per_100k.cbm")
submission.head()


## Results summary

The original experiments produced the following headline results:

| Experiment | R² |
|---|---:|
| Linear Regression, local test | 0.5964 |
| Random Forest, local test | 0.6305 |
| Gradient Boosting, local test | 0.6612 |
| CatBoost + `Deaths_per_100k`, local test | **0.8529** |
| CatBoost + `Deaths_per_100k`, 5-fold CV mean | **0.8364** |
| Final CatBoost Kaggle public leaderboard (30%) | **0.83616** |
| Final CatBoost Kaggle private leaderboard (70%) | **~0.65** |

The largest improvement came from feature representation rather than simply increasing neural-network depth. The explicit crude mortality-rate feature gave the boosted-tree model a normalized signal that was strongly useful for predicting the age-adjusted target.


### Generalization note

Although the engineered feature produced strong local cross-validation and public-leaderboard results, the approximately 0.65 private-leaderboard R² demonstrates that the model was less robust on the larger hidden holdout. This public–private gap is treated as an important result rather than omitted: strong random cross-validation and a strong public score did not guarantee equivalent performance on the full hidden distribution.
